In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F
import country_converter as coco
from itertools import chain

CLEANED_DATA_DIR = Path("../data/cleaned")
PROCESSED_DATA_DIR = Path("../data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{CLEANED_DATA_DIR}/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

df.show(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import seaborn as sns
import torch
import pyspark.sql.functions as F
from VehicleAutoencoder import VehicleAutoencoder

MODEL_CHECKPOINT_PATH = "best_vehicle_autoencoder.ckpt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VehicleAutoencoder.load_from_checkpoint(MODEL_CHECKPOINT_PATH)
model.to(device)
model.eval()

feature_cols = [
    "mass_in_running_order (kg)",
    "co2_emissions_WLTP (g/km)",
    "engine_capacity (cm3)",
    "engine_power (KW)",
    "electric_energy_consumption (Wh/km)"
]

df_sample = df.sample(withReplacement=False, fraction=0.15, seed=42)

is_electric = F.col("Motor energy") == "Electricity"
is_ice = F.col("Motor energy").isin("Petrol (excluding hybrids)", "Diesel (excluding hybrids)")

df_zeroed = df_sample.withColumn(
    "co2_emissions_WLTP (g/km)",
    F.when(is_electric, 0.0).otherwise(F.col("co2_emissions_WLTP (g/km)"))
).withColumn(
    "engine_capacity (cm3)",
    F.when(is_electric, 0.0).otherwise(F.col("engine_capacity (cm3)"))
).withColumn(
    "electric_energy_consumption (Wh/km)",
    F.when(is_electric & F.col("electric_energy_consumption (Wh/km)").isNull(), 0.0)
     .when(is_ice, 0.0)
     .otherwise(F.col("electric_energy_consumption (Wh/km)"))
)

df_clean = df_zeroed.dropna(subset=feature_cols)

pdf = df_clean.select(["Motor energy"] + feature_cols).toPandas()

X_raw = pdf[feature_cols].values.astype(np.float32)

mean = np.mean(X_raw, axis=0)
std = np.std(X_raw, axis=0)
std[std == 0.0] = 1.0

X_scaled = (X_raw - mean) / std
X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device)

with torch.no_grad():
    latent_coords = model.encode(X_tensor).cpu().numpy()

pdf["z_1"] = latent_coords[:, 0]
pdf["z_2"] = latent_coords[:, 1]

SAMPLES_PER_CLASS = 1000

pdf_balanced = (
    pdf.groupby("Motor energy", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLES_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)

sns.set_theme(style="whitegrid", font_scale=1.0)

g = sns.relplot(
    data=pdf_balanced,
    x="z_1",
    y="z_2",
    hue="Motor energy",
    col="Motor energy",
    col_wrap=3,
    kind="scatter",
    palette="tab10",
    alpha=0.5,
    s=25,
    height=3.5,
    aspect=1.2
)

g.set_axis_labels("Latent $z_1$", "Latent $z_2$")
g.set_titles(col_template="{col_name}")
g.figure.suptitle("Balanced 2D Autoencoder Projection", y=1.03, fontweight="bold", fontsize=14)

plt.show()